# Knowledge-Head Discovery (Instruct) — K-heads vs binding heads (Appendix M, Tables 18–19)

**Question (reviewer):** are the published binding heads (S-heads) really *gating* heads, or
just knowledge-retrieval heads? **Answer strategy:** run the *exact same* head-discovery
pipeline (L1-logistic-regression attention CV, 5-fold GroupKFold) on the KNOWLEDGE-PROBE
task, get an independent K-head set, and compare it with the S-heads: overlap/Jaccard +
near-miss diagnostics, group edge-knockout of the K-heads on BOTH tasks (primary/variant
subsets), and per-head knockouts.

**Reproduces:** Appendix M, Tables 18 and 19 of the paper.

**Provenance:** rebuilt from `khead_discovery_instruct_mistral.ipynb` (canonical v4 copy,
cells 0–28), parametrically consolidated with the three sibling copies
`khead_discovery_instruct_{llama,gemma,nemo}.ipynb` (per-model constants live in the
params cell below). Shared helpers are imported from `common/` (verbatim extractions
from the same canonical notebooks)..

**How to run:**
1. `export HF_TOKEN=...` before starting Jupyter (never hardcode it); make sure
   `data/N4_1k.pkl` is present at the repo root.
2. Set `MODEL_KEY` in the params cell (instruct Gemma uses the key `"gemma2"`).
3. Run top to bottom on a GPU (A100 recommended). Every expensive cell caches its
   output under `./results/<model>_instruct/` and skips itself on re-run.
   Runtime ≈ 30–45 min per model, dominated by cells 5 and 8.

**Validation gates (each prints PASS/FAIL diagnostics):**
- Cell 3: 847 pairs in both conditions; 3 fully decoded prompts for manual inspection.
- Cell 4: |ΔK| must reproduce the published value within ±5% (HARD STOP otherwise).
- Cell 5: feature matrix shape `[n_pairs, n_layers, n_heads, 3]`; NaN/span failures < 2%.
- Cell 6: mean CV AUC > 0.65, else the whole analysis is flagged INCONCLUSIVE.
- Cell 8: knockout mechanism sanity-checked on 5 prompts before the full pass.

**Decision rule:** Dissociation is supported only if (i) K-discovery AUC is adequate,
(ii) the K-head and S-head sets differ substantially (low Jaccard, not explained by
near-miss ranks), AND (iii) the 2×2 knockout matrix shows a double dissociation (K-heads
hit K more than S; S-heads hit S more than K in instruct). Otherwise the data support
shared infrastructure.

In [ ]:
MODEL_KEY = "mistral"  # one of {"mistral", "llama", "gemma2", "nemo"}

# -----------------------------------------------------------------------------
# Per-model constants, consolidated VERBATIM from the four canonical copies
#
#   PUBLISHED_ABS_DK   : published |dK| that the cell-4 gate must reproduce (+/-5%)
#   K_HEADS_PRIMARY    : group-knockout subset (cell 8), primary
#   K_HEADS_VARIANT    : group-knockout subset (cell 8), variant (None = skip)
#   HEADS_TO_TEST      : single-head KO list (standalone cell 8b)
#   HEADS_TO_TEST_BASE : single-head KO list, heads discovered on the BASE
#                        variant (second standalone 8b run; [] = the canonical
#                        copy for this model has no such run)
#
# NOTE (llama): the canonical llama copy contains TWO versions of cell 8. The
# first run used K_HEADS_VARIANT = [(7, 7), (8, 17), (11, 26)]; the final
# "extra run" — whose numbers are the ones reported in Tables 18–19 — used
# [(7, 7), (8, 17), (11, 14)]. The final (11, 14) version is retained below.
# -----------------------------------------------------------------------------
PER_MODEL = {
    "mistral": {
        "PUBLISHED_ABS_DK": 12.31,
        "K_HEADS_PRIMARY":  [(8, 16), (9, 23), (12, 9)],
        "K_HEADS_VARIANT":  None,                                  # L14H2 (+0.6% K) too small
        "HEADS_TO_TEST": [
            (11,7), (11,25), (12,13), (14,2), (14,13), (15,9), (15,30), (16,16), (17,24), (18,10),
            (19,29), (25,3)
        ],
        "HEADS_TO_TEST_BASE": [
            (9,23), (12,9)
        ],
    },
    "llama": {
        "PUBLISHED_ABS_DK": 4.90,
        "K_HEADS_PRIMARY":  [(7, 7), (8, 17)],
        # first-run alternative (documented, not used): [(7, 7), (8, 17), (11, 26)]
        "K_HEADS_VARIANT":  [(7, 7), (8, 17), (11, 14)],           # + K-not-binding L11H14 (final "extra run"; an earlier run used L11H26) (11h14!)
        "HEADS_TO_TEST": [
            (5,4),(6,14), (7,6), (7,7), (8,17), (8,18), (9,2), (9,23), (10,3), (10,30),
            (11,8), (11,26), (12,2), (12,6), (12,30),(13,13), (13,18), (14,7), (14,16),
            (15,15), (16,13), (22,22)
        ],
        "HEADS_TO_TEST_BASE": [
            (11,14)
        ],
    },
    "gemma2": {
        "PUBLISHED_ABS_DK": 8.69,
        "K_HEADS_PRIMARY":  [(11, 14), (13, 13)],
        "K_HEADS_VARIANT":  [(11, 14), (13, 13), (16, 15)],        # + binding-not-K L16H15
        "HEADS_TO_TEST": [
            (9, 2), (9, 12), (11, 14), (13, 9), (13, 12), (13, 13),
            (14, 10), (16, 3), (16, 10), (16, 15), (19, 14), (20, 2),
            (20, 4), (24, 6), (26, 2), (27, 6), (29, 12),
        ],
        "HEADS_TO_TEST_BASE": [
            (16, 14),
        ],
    },
    "nemo": {
        "PUBLISHED_ABS_DK": 4.57,
        "K_HEADS_PRIMARY":  [(8, 9), (10, 24)],
        "K_HEADS_VARIANT":  [(8, 9), (10, 24), (12, 15)],          # + K-not-binding L12H15
        "HEADS_TO_TEST": [
            (5,10), (8,9), (8,10), (8,20), (10,20),(10,24),(11,29), (12,6),(12,14),
            (12,15), (15,31), (16,28),(17,23), (18,24),(19,21),(22,20),
            (26,13)
        ],
        "HEADS_TO_TEST_BASE": [],  # the canonical nemo copy has no base-discovered 8b run
    },
}

# Model-keyed tables in the exact shape the verbatim experiment cells reference
# (PUBLISHED_ABS_DK[ACTIVE_MODEL], K_HEADS_PRIMARY.get(ACTIVE_MODEL), ...).
PUBLISHED_ABS_DK = {m: c["PUBLISHED_ABS_DK"] for m, c in PER_MODEL.items()}
K_HEADS_PRIMARY  = {m: c["K_HEADS_PRIMARY"]  for m, c in PER_MODEL.items()}
K_HEADS_VARIANT  = {m: c["K_HEADS_VARIANT"]  for m, c in PER_MODEL.items()}

In [ ]:
import sys, os
# Locate the repo root (the directory containing common/), whatever the kernel cwd
_p = os.path.abspath(".")
REPO_ROOT = _p if os.path.isdir(os.path.join(_p, "common")) else os.path.abspath("..")
assert os.path.isdir(os.path.join(REPO_ROOT, "common")), (
    "Cannot locate the repo root: run this notebook from its own directory or the repo root")
sys.path.insert(0, REPO_ROOT)

from common import config

CFG = config.init(MODEL_KEY, "instruct")
SEED = config.SEED
ACTIVE_MODEL = config.ACTIVE_MODEL   # historical alias used by the verbatim cells
DATA_DIR = config.DATA_DIR
OUTPUT_DIR = config.OUTPUT_DIR       # ./results/<model>_instruct/
HF_TOKEN = config.HF_TOKEN           # read from the HF_TOKEN environment variable

from common.text_parsers import extract_options
from common.instruct.data import (load_n4, build_factorial_as_conditions,
                                  build_knowledge_probes)
from common.instruct.prompts import (format_for_chat, find_option_token_ids,
                                     detect_spans, validate_spans)
# compute_logit_scores_edge / reset_eager_attention are deliberately NOT
# imported from common.instruct.hooks — this notebook defines its own verbatim
# khead-v4 copies in the machinery cell below (leak-window fidelity; see
from common.instruct.discovery import extract_attention_scores, run_outer_cv
from common.stats_utils import ttest_clustered

import time  # per-cell timers

# GPU report
import torch
if torch.cuda.is_available():
    _free, _total = torch.cuda.mem_get_info(0)
    print(f"GPU: {torch.cuda.get_device_name(0)}  free {_free/1e9:.1f} / {_total/1e9:.1f} GB")
else:
    print("!! CUDA not available — this notebook needs a GPU (A100 recommended)")

## Setup — imports and model load

The imports cell below is copied verbatim from the canonical copy. The model config
table, the text/data/prompt helpers, the edge-knockout machinery and the Stage-2 CV
utilities are imported from `common/` in the setup cell above (verbatim extractions
from the same canonical notebooks —. Model loading follows the
reference conventions: bf16, `device_map="auto"`, eager attention (the knockout
machinery patches `eager_attention_forward`; Gemma-2 soft-capping is handled inside
`edge_knockout` via `CFG["has_softcapping"]`).

In [ ]:
import os
import re
import gc
import ast
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import importlib
import matplotlib.pyplot as plt

from pathlib import Path
from collections import defaultdict, Counter
from contextlib import contextmanager
from itertools import combinations

from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from sklearn.utils import resample
from scipy import stats
from scipy.stats import ttest_rel

import warnings
warnings.filterwarnings('ignore')
import logging
logging.getLogger("transformers.generation.utils").setLevel(logging.ERROR)

## Edge-knockout machinery (verbatim khead v4 — deliberately notebook-local)

This notebook deliberately uses its own verbatim copy of the v4 knockout/reset machinery (leak-window semantics differ from common/*/hooks.py). Concretely: `edge_knockout` leaves its wrapper in `ALL_ATTENTION_FUNCTIONS._local_mapping` on exit and never clears `_EDGE_REGISTRY`; `reset_eager_attention` purges both **in this notebook's namespace** — the exact semantics the published Tables 18–19 were produced with. `compute_logit_scores_edge` is defined locally as well: the copy in `common.instruct.hooks` resolves `edge_knockout` inside its own module namespace, so importing it would silently bypass the notebook-local knockout defined here.

In [ ]:
# ================================================================
# EDGE-KNOCKOUT MACHINERY — verbatim khead v4, deliberately NOTEBOOK-LOCAL
# (NOT imported from common/instruct/hooks.py). The published Tables 18-20
# were produced with THIS version: edge_knockout leaves its wrapper in
# ALL_ATTENTION_FUNCTIONS._local_mapping on exit and never clears
# _EDGE_REGISTRY; reset_eager_attention purges both in this notebook's
# namespace. compute_logit_scores_edge is defined here too so that its
# edge_knockout call resolves to the notebook-local version (the common/
# ================================================================
# ================================================================
# UNIFIED EDGE KNOCKOUT (supports soft-capping for Gemma-2)
# ================================================================

_EDGE_REGISTRY = {}

@contextmanager
def edge_knockout(model, heads, source_tokens, target_tokens):
    """Zero attention edges from source → target for specific heads.
    Automatically handles Gemma-2 soft-capping via CFG['has_softcapping'].
    """
    global _EDGE_REGISTRY
    if not heads or not source_tokens or not target_tokens:
        yield; return

    attn_module = importlib.import_module(CFG["attn_module"])
    from transformers.modeling_utils import ALL_ATTENTION_FUNCTIONS

    _EDGE_REGISTRY = {
        'heads': {k: set(v) for k, v in heads.items()},
        'source_tokens': sorted(set(source_tokens)),
        'target_tokens': sorted(set(target_tokens)),
    }

    original_eager = attn_module.eager_attention_forward
    n_heads = model.config.num_attention_heads
    n_kv_heads = getattr(model.config, 'num_key_value_heads', n_heads)
    kv_group_size = n_heads // n_kv_heads
    head_dim = getattr(model.config, "head_dim", None) or (
        model.config.hidden_size // n_heads)
    softcapping = getattr(model.config, 'attn_logit_softcapping', None) \
                  if CFG["has_softcapping"] else None

    def wrapped_eager(module, query, key, value, attention_mask, **kwargs):
        scaling = kwargs.pop('scaling', None)
        dropout = kwargs.pop('dropout', 0.0)
        layer_idx = getattr(module, 'layer_idx', None)
        heads_to_edit = _EDGE_REGISTRY['heads'].get(layer_idx, None)
        if not heads_to_edit:
            return original_eager(module, query, key, value, attention_mask,
                                  scaling=scaling, dropout=dropout, **kwargs)

        attn_output, attn_weights = original_eager(
            module, query, key, value, attention_mask,
            scaling=scaling, dropout=dropout, **kwargs)
        attn_output = attn_output.clone()

        # Detect head axis
        if query.shape[1] == n_heads:   qkv_head_axis = 1
        elif query.shape[2] == n_heads: qkv_head_axis = 2
        else: return attn_output, attn_weights

        if attn_output.shape[1] == n_heads:   out_head_axis = 1
        elif attn_output.shape[2] == n_heads: out_head_axis = 2
        else: return attn_output, attn_weights

        seq_len_q = query.shape[2] if qkv_head_axis == 1 else query.shape[1]
        seq_len_k = key.shape[2] if qkv_head_axis == 1 else key.shape[1]
        src = [s for s in _EDGE_REGISTRY['source_tokens'] if s < seq_len_k]
        tgt = [t for t in _EDGE_REGISTRY['target_tokens'] if t < seq_len_q]
        if not src or not tgt:
            return attn_output, attn_weights

        for h in heads_to_edit:
            kv_h = h // kv_group_size
            if qkv_head_axis == 1:
                q_h, k_h, v_h = query[:, h, :, :], key[:, kv_h, :, :], value[:, kv_h, :, :]
            else:
                q_h, k_h, v_h = query[:, :, h, :], key[:, :, kv_h, :], value[:, :, kv_h, :]

            attn_scores = torch.matmul(q_h, k_h.transpose(-2, -1)) * scaling
            # ── Gemma-2 soft-capping ──
            if softcapping is not None:
                attn_scores = attn_scores / softcapping
                attn_scores = torch.tanh(attn_scores)
                attn_scores = attn_scores * softcapping
            if attention_mask is not None and attention_mask.dim() == 4:
                if attention_mask.shape[1] == 1:
                    attn_scores = attn_scores + attention_mask[:, 0, :seq_len_q, :seq_len_k]
                else:
                    attn_scores = attn_scores + attention_mask[:, h, :seq_len_q, :seq_len_k]
            # Zero the target edges
            for t_idx in tgt:
                for s_idx in src:
                    attn_scores[:, t_idx, s_idx] = torch.finfo(attn_scores.dtype).min
            attn_probs = F.softmax(attn_scores, dim=-1)
            new_output = torch.matmul(attn_probs, v_h)
            if out_head_axis == 1:
                attn_output[:, h, :, :] = new_output
            else:
                attn_output[:, :, h, :] = new_output

        return attn_output, attn_weights

    # Monkey-patch
    attn_module.eager_attention_forward = wrapped_eager
    orig_entry = None
    try:
        orig_entry = ALL_ATTENTION_FUNCTIONS.get('eager', None)
        ALL_ATTENTION_FUNCTIONS['eager'] = wrapped_eager
    except: pass
    try:
        yield
    finally:
        attn_module.eager_attention_forward = original_eager
        if orig_entry is not None:
            try: ALL_ATTENTION_FUNCTIONS['eager'] = orig_entry
            except: pass


# ================================================================
# UNIFIED EDGE SCALE (α-scaling for dose-response + DiffAware)
# ================================================================

_SCALE_REGISTRY = {}

@contextmanager
def edge_scale(model, heads, source_tokens, target_tokens, alpha=0.0):
    """Scale attention edges: α=0 → full KO, α=1 → baseline, α>1 → amplify."""
    global _SCALE_REGISTRY
    if not heads or not source_tokens or not target_tokens or alpha == 1.0:
        yield; return

    attn_module = importlib.import_module(CFG["attn_module"])
    from transformers.modeling_utils import ALL_ATTENTION_FUNCTIONS

    _SCALE_REGISTRY = {
        'heads': {k: set(v) for k, v in heads.items()},
        'source_tokens': sorted(set(source_tokens)),
        'target_tokens': sorted(set(target_tokens)),
        'alpha': alpha,
    }

    original_eager = attn_module.eager_attention_forward
    n_heads = model.config.num_attention_heads
    n_kv_heads = getattr(model.config, 'num_key_value_heads', n_heads)
    kv_group_size = n_heads // n_kv_heads
    head_dim = getattr(model.config, "head_dim", None) or (
        model.config.hidden_size // n_heads)
    softcapping = getattr(model.config, 'attn_logit_softcapping', None) \
                  if CFG["has_softcapping"] else None

    def wrapped_eager(module, query, key, value, attention_mask, **kwargs):
        scaling = kwargs.pop('scaling', None)
        dropout = kwargs.pop('dropout', 0.0)
        layer_idx = getattr(module, 'layer_idx', None)
        heads_to_edit = _SCALE_REGISTRY['heads'].get(layer_idx, None)
        if not heads_to_edit:
            return original_eager(module, query, key, value, attention_mask,
                                  scaling=scaling, dropout=dropout, **kwargs)

        alpha_val = _SCALE_REGISTRY['alpha']

        # Normal forward
        attn_output, attn_weights = original_eager(
            module, query, key, value, attention_mask,
            scaling=scaling, dropout=dropout, **kwargs)
        attn_output = attn_output.clone()

        if query.shape[1] == n_heads:   qkv_head_axis = 1
        elif query.shape[2] == n_heads: qkv_head_axis = 2
        else: return attn_output, attn_weights

        if attn_output.shape[1] == n_heads:   out_head_axis = 1
        elif attn_output.shape[2] == n_heads: out_head_axis = 2
        else: return attn_output, attn_weights

        seq_len_q = query.shape[2] if qkv_head_axis == 1 else query.shape[1]
        seq_len_k = key.shape[2] if qkv_head_axis == 1 else key.shape[1]
        src = [s for s in _SCALE_REGISTRY['source_tokens'] if s < seq_len_k]
        tgt = [t for t in _SCALE_REGISTRY['target_tokens'] if t < seq_len_q]
        if not src or not tgt:
            return attn_output, attn_weights

        for h in heads_to_edit:
            kv_h = h // kv_group_size
            if qkv_head_axis == 1:
                q_h, k_h, v_h = query[:, h, :, :], key[:, kv_h, :, :], value[:, kv_h, :, :]
            else:
                q_h, k_h, v_h = query[:, :, h, :], key[:, :, kv_h, :], value[:, :, kv_h, :]

            attn_scores = torch.matmul(q_h, k_h.transpose(-2, -1)) * scaling
            if softcapping is not None:
                attn_scores = attn_scores / softcapping
                attn_scores = torch.tanh(attn_scores)
                attn_scores = attn_scores * softcapping
            if attention_mask is not None and attention_mask.dim() == 4:
                if attention_mask.shape[1] == 1:
                    attn_scores = attn_scores + attention_mask[:, 0, :seq_len_q, :seq_len_k]
                else:
                    attn_scores = attn_scores + attention_mask[:, h, :seq_len_q, :seq_len_k]

            # ── KO version: zero target edges ──
            attn_scores_ko = attn_scores.clone()
            for t_idx in tgt:
                for s_idx in src:
                    attn_scores_ko[:, t_idx, s_idx] = torch.finfo(attn_scores_ko.dtype).min
            attn_probs_ko = F.softmax(attn_scores_ko, dim=-1)
            out_ko = torch.matmul(attn_probs_ko, v_h)

            # ── Normal version ──
            attn_probs_normal = F.softmax(attn_scores, dim=-1)
            out_normal = torch.matmul(attn_probs_normal, v_h)

            # ── Blend: out = out_ko + α * (out_normal - out_ko) ──
            new_output = out_ko + alpha_val * (out_normal - out_ko)

            if out_head_axis == 1:
                attn_output[:, h, :, :] = new_output
            else:
                attn_output[:, :, h, :] = new_output

        return attn_output, attn_weights

    attn_module.eager_attention_forward = wrapped_eager
    orig_entry = None
    try:
        orig_entry = ALL_ATTENTION_FUNCTIONS.get('eager', None)
        ALL_ATTENTION_FUNCTIONS['eager'] = wrapped_eager
    except: pass
    try:
        yield
    finally:
        attn_module.eager_attention_forward = original_eager
        if orig_entry is not None:
            try: ALL_ATTENTION_FUNCTIONS['eager'] = orig_entry
            except: pass


# ================================================================
# UNIFIED LOGIT SCORING (with optional edge intervention)
# ================================================================

def compute_logit_scores_edge(model, tokenizer, formatted_texts, raw_texts,
                               heads, positions_list, edge_mode, option_tokens,
                               return_lps=False):
    """Compute S = logP(c) - logsumexp(logP(a), logP(b)) with optional edge KO.
    If return_lps=True, also returns a list of per-option logprob dicts."""
    scores = []
    all_lps = []
    first_device = next(model.parameters()).device
    for i, text in enumerate(formatted_texts):
        enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        enc = {k: v.to(first_device) for k, v in enc.items()}
        pos = positions_list[i]

        if not heads or pos is None:
            with torch.no_grad():
                outputs = model(**enc)
        else:
            if edge_mode == 'B_to_item':
                src, tgt = pos['item_tokens'], pos['B_tokens']
            elif edge_mode == 'A_to_item':
                src, tgt = pos['item_tokens'], pos['A_tokens']
            else:
                src, tgt = [], []
            with torch.no_grad(), edge_knockout(model, heads, src, tgt):
                outputs = model(**enc)

        logits = outputs.logits[0, -1, :].float()
        lp = F.log_softmax(logits, dim=-1)
        lp_a = torch.logsumexp(lp[option_tokens['a']], dim=0).item()
        lp_b = torch.logsumexp(lp[option_tokens['b']], dim=0).item()
        lp_c = torch.logsumexp(lp[option_tokens['c']], dim=0).item()
        scores.append(lp_c - torch.logsumexp(torch.tensor([lp_a, lp_b]), dim=0).item())
        if return_lps:
            all_lps.append({'a': lp_a, 'b': lp_b, 'c': lp_c})
        del enc, outputs, logits, lp
        if i % 100 == 0 and i > 0:
            torch.cuda.empty_cache()
    return (np.array(scores), all_lps) if return_lps else np.array(scores)


def compute_scores_scaled(model, tokenizer, formatted_texts, positions_list,
                           heads_dict, edge_mode, option_tokens, alpha=1.0):
    """Compute S-scores with edge_scale (α-scaling)."""
    scores = []
    first_device = next(model.parameters()).device
    for i, text in enumerate(formatted_texts):
        enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        enc = {k: v.to(first_device) for k, v in enc.items()}
        pos = positions_list[i]

        if alpha == 1.0 or pos is None or not heads_dict:
            with torch.no_grad():
                outputs = model(**enc)
        else:
            if edge_mode == 'B_to_item':
                src, tgt = pos['item_tokens'], pos['B_tokens']
            elif edge_mode == 'A_to_item':
                src, tgt = pos['item_tokens'], pos['A_tokens']
            else:
                src, tgt = [], []
            with torch.no_grad(), edge_scale(model, heads_dict, src, tgt, alpha=alpha):
                outputs = model(**enc)

        logits = outputs.logits[0, -1, :].float()
        lp = F.log_softmax(logits, dim=-1)
        lp_a = torch.logsumexp(lp[option_tokens['a']], dim=0).item()
        lp_b = torch.logsumexp(lp[option_tokens['b']], dim=0).item()
        lp_c = torch.logsumexp(lp[option_tokens['c']], dim=0).item()
        scores.append(lp_c - torch.logsumexp(torch.tensor([lp_a, lp_b]), dim=0).item())
        del enc, outputs, logits, lp
        if i % 100 == 0 and i > 0:
            torch.cuda.empty_cache()
    return np.array(scores)


# ── reset_eager_attention — verbatim from the canonical cell 8 (causal
# cross-test, khead v4); operates on THIS notebook's _EDGE_REGISTRY. ──
from transformers.modeling_utils import ALL_ATTENTION_FUNCTIONS as _AAF


def reset_eager_attention():
    """Purge the wrapper edge_knockout leaks into ALL_ATTENTION_FUNCTIONS
    and clear the stale edge registry (see header note)."""
    global _EDGE_REGISTRY
    if hasattr(_AAF, '_local_mapping'):
        _AAF._local_mapping.pop('eager', None)
    _EDGE_REGISTRY = {}

In [ ]:
# ================================================================
# CELL 2 (load) — MODEL + TOKENIZER (verbatim conventions from the
# reference: bf16, device_map="auto", eager attention — the knockout
# machinery patches eager_attention_forward, and Gemma-2 soft-capping
# is handled inside edge_knockout via CFG["has_softcapping"]).
# ================================================================
_t_cell = time.time()

print(f"Loading {CFG['model_path']} ...")
tokenizer = AutoTokenizer.from_pretrained(
    CFG['model_path'], trust_remote_code=True, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    CFG['model_path'], dtype=torch.bfloat16, device_map="auto",
    trust_remote_code=True, token=HF_TOKEN,
    attn_implementation="eager",
)
model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

N_LAYERS = model.config.num_hidden_layers
N_HEADS = model.config.num_attention_heads
print(f"  {N_LAYERS} layers x {N_HEADS} heads")

option_tokens_raw = find_option_token_ids(tokenizer)
first_device = next(model.parameters()).device
option_tokens = {opt: torch.tensor(ids, device=first_device)
                 for opt, ids in option_tokens_raw.items()}
for opt, ids in option_tokens_raw.items():
    decoded = [tokenizer.decode([i]) for i in ids]
    print(f"  option '{opt}': {decoded}")

print(f"\n[cell 2 load done in {time.time()-_t_cell:.1f}s]")

# Expose the runtime singletons to the shared helper modules (common/config.py):
# discovery.extract_attention_scores reads config.first_device. (The edge-
# knockout machinery is notebook-local here — verbatim khead v4 cell above —
# and reads the local CFG alias, not config.CFG.)
config.model = model
config.tokenizer = tokenizer
config.first_device = first_device

## Cell 3 — Build the KNOWLEDGE-PROBE prompts (+ GATE: 847 pairs, 3 decoded prompts)

In [ ]:
# ================================================================
# CELL 3 — KNOWLEDGE-PROBE PROMPTS
# Same 847 factorial pairs as the binding task; the scenario is replaced
# by the direct association question. Construction is the verbatim
# build_knowledge_probes from the reference notebook (R randomized over
# a/b via the factorial assoc_pos, no-assoc fixed at (c)).
# ================================================================
_t_cell = time.time()

print("=" * 80)
print(f"CELL 3: KNOWLEDGE-PROBE PROMPTS — {CFG['label']}")
print("=" * 80)

cultural_items, neutral_items = load_n4(DATA_DIR)
data = build_factorial_as_conditions(cultural_items, seed=SEED)
n_total = len(data['B_cult'])
conditions = ['B_cult', 'B_unrel']

knowledge = build_knowledge_probes(data)  # verbatim helper from the reference
k_conditions = ['K_cult', 'K_unrel']

# ── GATE 3.1: pair counts ──
assert n_total == 847, f"GATE 3.1 FAILED: expected 847 factorial pairs, got {n_total}"
assert len(knowledge['K_cult']) == 847 and len(knowledge['K_unrel']) == 847, \
    "GATE 3.1 FAILED: knowledge probes != 847 in one of the conditions"
n_items_unique = len(set(data['items_cult']))
print(f"\n  [GATE 3.1 OK] 847 pairs in both K conditions, {n_items_unique} unique items")

# ── Format with the model's chat template (verbatim format_for_chat) ──
texts_fmt_k = {c: format_for_chat(knowledge[c], tokenizer) for c in k_conditions}
texts_fmt = {c: format_for_chat(data[c], tokenizer) for c in conditions}  # binding, for cells 5/8

# ── GATE 3.2: print 3 fully decoded prompts (match + mismatch) ──
print("\n  [GATE 3.2] 3 fully decoded K prompts (tokenize -> decode: exactly what the model sees):")
for i in range(3):
    for cond in k_conditions:
        ids = tokenizer(texts_fmt_k[cond][i], return_tensors="pt")["input_ids"][0]
        print(f"\n  ----- pair {i} | {cond} | item = {data['items_cult'][i]} "
              f"| assoc_pos = {data['assoc_pos'][i]} -----")
        print("  | " + tokenizer.decode(ids).replace("\n", "\n  | "))
print("\n  Manually check: question = 'Which of the following cultural identities is most")
print("  associated with <item>?', options (a)/(b) = identities, (c) = 'Neither is")
print("  particularly associated'; the match condition contains the associated identity")
print("  at assoc_pos, the mismatch condition has it swapped out.")

print(f"\n[cell 3 done in {time.time()-_t_cell:.1f}s]")

## Cell 4 — Baseline K-scores (+ GATE: |ΔK| must reproduce the published value ±5%, HARD STOP)

In [ ]:
# ================================================================
# CELL 4 — BASELINE K-SCORES on both conditions
# K = lp_c - logsumexp(lp_a, lp_b), identical to the reference notebook's
# knowledge evaluation. Cached to results/; skipped if the cache exists.
# GATE: |dK| must match the published instruct value within +/-5%,
# otherwise we HARD STOP — everything downstream depends on prompt fidelity.
# ================================================================
_t_cell = time.time()

# PUBLISHED_ABS_DK is defined in the params cell (PER_MODEL consolidation).
DK_REL_TOL = 0.05  # +/-5%

K_BASELINE_CACHE = OUTPUT_DIR / f"{ACTIVE_MODEL}_instruct_K_baseline_scores.pkl"

if K_BASELINE_CACHE.exists():
    with open(K_BASELINE_CACHE, "rb") as f:
        results_kscore = pickle.load(f)
    print(f"  [cache hit] {K_BASELINE_CACHE.name} — skipping recomputation")
else:
    results_kscore = {}
    for cond in k_conditions:
        scores, c_chosen, p_c_list, lps = [], [], [], []
        for i, text in enumerate(texts_fmt_k[cond]):
            enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
            enc = {k: v.to(first_device) for k, v in enc.items()}
            with torch.no_grad():
                outputs = model(**enc)
            logits = outputs.logits[0, -1, :].float()
            lp = F.log_softmax(logits, dim=-1)
            lp_a = torch.logsumexp(lp[option_tokens['a']], dim=0).item()
            lp_b = torch.logsumexp(lp[option_tokens['b']], dim=0).item()
            lp_c = torch.logsumexp(lp[option_tokens['c']], dim=0).item()
            K = lp_c - torch.logsumexp(torch.tensor([lp_a, lp_b]), dim=0).item()
            scores.append(K)
            chosen = 'c' if (lp_c >= lp_a and lp_c >= lp_b) else ('a' if lp_a >= lp_b else 'b')
            c_chosen.append(1 if chosen == 'c' else 0)
            lp_all = torch.logsumexp(torch.tensor([lp_a, lp_b, lp_c]), dim=0).item()
            p_c_list.append(np.exp(lp_c - lp_all))
            lps.append({'a': lp_a, 'b': lp_b, 'c': lp_c})
            del enc, outputs, logits, lp
            if i % 200 == 0 and i > 0:
                print(f"    {cond}: {i}/{n_total}  ({time.time()-_t_cell:.0f}s)")
                torch.cuda.empty_cache()
        results_kscore[cond] = {
            'K': np.array(scores), 'c_chosen': np.array(c_chosen),
            'p_c': np.array(p_c_list), 'logprobs': lps,
        }
    with open(K_BASELINE_CACHE, "wb") as f:
        pickle.dump(results_kscore, f)
    print(f"  Saved {K_BASELINE_CACHE}")

# ── Diagnostics ──
print(f"\n  {'':10s}  {'Mean K':>8s}  {'Med K':>8s}  {'(c) rate':>8s}  {'Mean P(c)':>9s}")
for cond in k_conditions:
    r = results_kscore[cond]
    print(f"  {cond:10s}  {r['K'].mean():8.3f}  {np.median(r['K']):8.3f}  "
          f"{r['c_chosen'].mean():8.3f}  {r['p_c'].mean():9.3f}")

delta_K = results_kscore['K_cult']['K'].mean() - results_kscore['K_unrel']['K'].mean()
target_dK = PUBLISHED_ABS_DK[ACTIVE_MODEL]
rel_dev = abs(abs(delta_K) - target_dK) / target_dK
print(f"\n  delta(K) = {delta_K:.4f}   |dK| = {abs(delta_K):.4f}")
print(f"  published |dK| ({ACTIVE_MODEL} instruct) = {target_dK}   rel. deviation = {100*rel_dev:.2f}%")

# ── GATE 4: HARD STOP if |dK| does not reproduce ──
if rel_dev > DK_REL_TOL:
    raise RuntimeError(
        f"GATE 4 FAILED: |dK| = {abs(delta_K):.4f} deviates {100*rel_dev:.1f}% from the "
        f"published {target_dK} (tolerance +/-5%). The K prompts do NOT reproduce the "
        f"published knowledge baseline — check ACTIVE_MODEL, the chat template, the N4 "
        f"file and the option-token mapping BEFORE running anything downstream. "
        f"All later cells are invalid until this gate passes.")
print(f"  [GATE 4 OK] |dK| reproduces the published value within +/-5%")

print(f"\n[cell 4 done in {time.time()-_t_cell:.1f}s]")

## Cell 5 — Attention extraction on K-prompts (THE expensive cell; cached) (+ GATE: shape `[n_pairs, n_layers, n_heads, 3]`, NaN/span failures < 2%)

Stage-2 definitions (`extract_attention_scores` + CV utilities) are imported from `common/instruct/discovery.py` — copied **verbatim** from the reference. The cell below is the extraction driver. Feature semantics are unchanged: per head, `bind_a_to_item` = attention from the identity-in-(a) span (queries, MEAN) to the item span (keys, SUM); same for (b); `bind_avg` = mean of the two. For the K task the identity spans live in the options and the item span lives in the question — `detect_spans` handles both, as it already does for the knowledge knockout in the reference.

The S-task features are ALSO extracted here (to a separate new cache) so that cell 6 can (a) re-run the binding CV as a positive control and (b) compute the vice-versa near-miss ranks in cell 7. Set `RUN_S_CV = False` to skip and halve the cost.

In [ ]:
# ================================================================
# CELL 5 (driver) — ATTENTION EXTRACTION, cached to results/
# Replicates the reference Stage-2 extraction loop exactly (prompts are
# chat-formatted WITHOUT the answer suffix, as in the reference cell 12;
# spans detected with detect_spans, item_required=True).
# ================================================================
_t_cell = time.time()

RUN_S_CV = True  # S-task features: positive control + vice-versa near-miss (cell 7)

K_ATTN_CACHE = OUTPUT_DIR / f"{ACTIVE_MODEL}_instruct_K_attn_features.pkl"
S_ATTN_CACHE = OUTPUT_DIR / f"{ACTIVE_MODEL}_instruct_S_attn_features_khead.pkl"


def extract_task_features(task_label, task_texts_raw, task_conds, cache_path):
    """Run the verbatim Stage-2 extraction loop on one task (K or S)."""
    if cache_path.exists():
        with open(cache_path, "rb") as f:
            out = pickle.load(f)
        print(f"  [cache hit] {cache_path.name}: "
              f"{len(out['valid_indices'])} valid pairs, "
              f"{len(out['skipped_indices'])} skipped — skipping recomputation")
        return out

    all_features = {c: [] for c in task_conds}
    valid_indices, scenarios_valid, skipped_indices = [], [], []

    for i in range(n_total):
        item_cult = data['items_cult'][i]
        cond_info, prompts, spans_all = {}, {}, {}
        all_ok = True
        for cond in task_conds:
            full_text = task_texts_raw[cond][i]
            q_text = full_text.split("\n\n")[0]
            oa, ob = extract_options(q_text)
            cond_info[cond] = {'question': q_text, 'opt_a': oa, 'opt_b': ob}
            # Chat formatting exactly as the reference Stage-2 loop (no answer suffix)
            if CFG["chat_format"] == "gemma":
                prompt = tokenizer.apply_chat_template(
                    [{"role": "user", "content": " " + full_text}],
                    tokenize=False, add_generation_prompt=True)
            else:
                prompt = tokenizer.apply_chat_template(
                    [{"role": "user", "content": full_text}],
                    tokenize=False, add_generation_prompt=True)
            enc = tokenizer(prompt, return_tensors="pt")
            ids = enc["input_ids"][0].tolist()
            sp = detect_spans(ids, q_text, oa, ob, item_cult, tokenizer,
                              item_required=True)
            if sp is None:
                all_ok = False
                del enc
                break
            prompts[cond] = prompt
            spans_all[cond] = sp
            del enc
        if not all_ok:
            skipped_indices.append(i)
            continue
        if len(valid_indices) < 3:  # span validation printout for the first 3 pairs
            for cond in task_conds:
                enc = tokenizer(prompts[cond], return_tensors="pt")
                validate_spans(enc["input_ids"][0], spans_all[cond], tokenizer,
                               label=f"{task_label} {cond} pair {i}: "
                                     f"{cond_info[cond]['opt_a']} vs {cond_info[cond]['opt_b']}")
                del enc
        for cond in task_conds:
            scores = extract_attention_scores(model, tokenizer, prompts[cond], spans_all[cond])
            all_features[cond].append(scores)
        valid_indices.append(i)
        scenarios_valid.append(data['scenarios'][i])
        if len(valid_indices) % 100 == 0:
            print(f"    {task_label}: {len(valid_indices)} valid / {i+1} seen "
                  f"(skipped {len(skipped_indices)})  [{time.time()-_t_cell:.0f}s]")

    out = {
        'features': all_features,
        'valid_indices': valid_indices,
        'scenarios_valid': scenarios_valid,
        'skipped_indices': skipped_indices,
        'meta': {'model': ACTIVE_MODEL, 'variant': 'instruct', 'task': task_label,
                 'n_total': n_total, 'seed': SEED,
                 'n_layers': N_LAYERS, 'n_heads': N_HEADS},
    }
    with open(cache_path, "wb") as f:
        pickle.dump(out, f)
    print(f"  Saved {cache_path}  ({os.path.getsize(cache_path)/1e6:.1f} MB)")
    return out


def stack_features(feat_list):
    """[n_pairs, n_layers, n_heads, 3] with feature order (f_a, f_b, f_avg)."""
    return np.stack([
        np.stack([f['bind_a_to_item'], f['bind_b_to_item'], f['bind_avg']], axis=-1)
        for f in feat_list])


def gate_features(attn, task_conds, label, hard=True):
    """GATE 5: shape + NaN + span-failure rate (< 2% of pairs)."""
    n_valid = len(attn['valid_indices'])
    n_skip = len(attn['skipped_indices'])
    worst_nan = 0
    for cond in task_conds:
        stk = stack_features(attn['features'][cond])
        expected = (n_valid, N_LAYERS, N_HEADS, 3)
        assert stk.shape == expected, \
            f"GATE 5 FAILED [{label}/{cond}]: feature shape {stk.shape} != {expected}"
        n_nan = int(np.isnan(stk).any(axis=(1, 2, 3)).sum())
        worst_nan = max(worst_nan, n_nan)
        print(f"  [{label}/{cond}] feature matrix {stk.shape}  NaN pairs: {n_nan}")
    fail_rate = (n_skip + worst_nan) / n_total
    print(f"  [{label}] span failures: {n_skip}/{n_total} ({100*n_skip/n_total:.2f}%)"
          f"   excluded pair indices: {attn['skipped_indices'] if n_skip else '[]'}")
    msg = (f"GATE 5 [{label}]: NaN/span-failure rate = {100*fail_rate:.2f}% "
           f"(must be < 2%)")
    if fail_rate < 0.02:
        print(f"  [GATE 5 OK] {msg}")
    elif hard:
        raise RuntimeError("GATE 5 FAILED: " + msg)
    else:
        print("  !! WARNING (soft gate): " + msg)


print("=" * 80)
print(f"CELL 5: ATTENTION EXTRACTION — {CFG['label']}")
print("=" * 80)

print("\n  K-task (knowledge probes):")
attn_K = extract_task_features("K-task", knowledge, k_conditions, K_ATTN_CACHE)
gate_features(attn_K, k_conditions, "K-task", hard=True)

if RUN_S_CV:
    print("\n  S-task (binding — NEW cache, original binding caches untouched):")
    attn_S = extract_task_features("S-task", data, conditions, S_ATTN_CACHE)
    gate_features(attn_S, conditions, "S-task", hard=False)
else:
    attn_S = None
    print("\n  RUN_S_CV = False — skipping S-task extraction "
          "(vice-versa near-miss in cell 7 will be unavailable)")

print(f"\n[cell 5 done in {time.time()-_t_cell:.1f}s]")

## Cell 6 — Head discovery: Stage-2 CV on the K task (+ GATE: mean AUC > 0.65; stable set = selected in ≥3/5 folds)

In [ ]:
# ================================================================
# CELL 6 — HEAD DISCOVERY (verbatim Stage-2 CV, run on the K features)
# L1 LogisticRegression (penalty='l1', solver='liblinear'), 5-fold
# GroupKFold grouped by cultural item, C from [0.001..10] by inner CV,
# ROC-AUC scoring. Stability: head selected in >= 3 of 5 folds.
# ================================================================
_t_cell = time.time()

FEATURE_NAMES = ['bind_avg', 'bind_a_to_item', 'bind_b_to_item']
MIN_FOLDS_STABLE = 3
AUC_GATE = 0.65


def run_task_cv(attn, task_conds, task_label):
    """Alias the task's (match, mismatch) conditions onto the 'B_cult'/'B_unrel'
    keys that the verbatim Stage-2 code expects (match -> y=1, mismatch -> y=0)."""
    feats = {'B_cult': attn['features'][task_conds[0]],
             'B_unrel': attn['features'][task_conds[1]]}
    cv = {}
    for feat in FEATURE_NAMES:
        print(f"\n  ### {task_label} CV — feature '{feat}' ###")
        folds, summary = run_outer_cv(feats, attn['scenarios_valid'], feature_name=feat)
        cv[feat] = {'folds': folds, 'summary': summary}
    return cv


def stable_set(cv, feat='bind_avg', min_folds=MIN_FOLDS_STABLE):
    s = cv[feat]['summary']
    if not s:
        return []
    return sorted(tuple(lh) for lh, cnt in s.get('stable_heads', []) if cnt >= min_folds)


print("=" * 80)
print(f"K-TASK HEAD DISCOVERY — {CFG['label']}")
print("=" * 80)
cv_K = run_task_cv(attn_K, k_conditions, "K-task")

if attn_S is not None:
    print("\n" + "=" * 80)
    print(f"S-TASK CV (positive control + vice-versa near-miss) — {CFG['label']}")
    print("=" * 80)
    cv_S = run_task_cv(attn_S, conditions, "S-task")
else:
    cv_S = None

# ── Primary K-head set: bind_avg, >= 3/5 folds ──
K_HEADS_LIST = stable_set(cv_K, 'bind_avg')
K_HEADS_STRICT = stable_set(cv_K, 'bind_avg', min_folds=4)

# ── GATE 6: discovery quality ──
fold_aucs = [f['test_auc'] for f in cv_K['bind_avg']['folds']]
auc_K = float(np.mean(fold_aucs)) if fold_aucs else float('nan')
print("\n[GATE 6] K-task discovery quality (bind_avg):")
print(f"  per-fold test AUC: {['%.3f' % a for a in fold_aucs]}")
print(f"  mean test AUC = {auc_K:.3f}  (gate: > {AUC_GATE})")
K_DISCOVERY_WEAK = not (auc_K > AUC_GATE)
if K_DISCOVERY_WEAK:
    print("  " + "!" * 72)
    print("  !! WARNING: K-head discovery is WEAK on this model (mean AUC <= 0.65).")
    print("  !! The K-head set is unreliable; the overlap analysis (cells 7-9) should")
    print("  !! be treated as INCONCLUSIVE for this model.")
    print("  " + "!" * 72)
else:
    print("  [GATE 6 OK] discovery AUC is adequate")

print(f"\n  K-heads  (>= {MIN_FOLDS_STABLE}/5 folds, bind_avg): "
      f"{[f'L{l}H{h}' for l, h in K_HEADS_LIST]}")
print(f"  K-heads  (>= 4/5 folds, strict):        "
      f"{[f'L{l}H{h}' for l, h in K_HEADS_STRICT]}")
for feat in FEATURE_NAMES[1:]:
    alt = stable_set(cv_K, feat)
    print(f"  [context] {feat:>15s} stable set: {[f'L{l}H{h}' for l, h in alt]}")

# ── Positive control: the S-CV should re-find the published S-heads ──
if cv_S is not None:
    S_FOUND = stable_set(cv_S, 'bind_avg')
    _pub = sorted((l, h) for l, hs in CFG['heads'].items() for h in hs)
    _missing = [lh for lh in _pub if lh not in S_FOUND]
    print(f"\n  [positive control] S-task stable set (bind_avg): "
          f"{[f'L{l}H{h}' for l, h in S_FOUND]}")
    if _missing:
        print(f"  !! WARNING: published S-heads not re-found by the S-CV: "
              f"{[f'L{l}H{h}' for l, h in _missing]}")
        print("  !! (published heads were finalized with the causal Stage-3 filter on top")
        print("  !!  of the CV, so treat this as a soft check — but do inspect the fold table)")
    else:
        print("  [OK] all published S-heads re-discovered on the binding task")

with open(OUTPUT_DIR / f"{ACTIVE_MODEL}_instruct_khead_cv.pkl", "wb") as f:
    pickle.dump({'cv_K': cv_K, 'cv_S': cv_S,
                 'K_HEADS_LIST': K_HEADS_LIST, 'K_HEADS_STRICT': K_HEADS_STRICT,
                 'auc_K': auc_K, 'fold_aucs': fold_aucs,
                 'K_DISCOVERY_WEAK': K_DISCOVERY_WEAK}, f)
print(f"\n  Saved CV results to {OUTPUT_DIR / (ACTIVE_MODEL + '_instruct_khead_cv.pkl')}")

print(f"\n[cell 6 done in {time.time()-_t_cell:.1f}s]")

## Cell 7 — Overlap analysis: K-heads vs published S-heads (Jaccard + near-miss + layer bands)

In [ ]:
# ================================================================
# CELL 7 — OVERLAP ANALYSIS
# Hardcoded published S-heads; intersection / Jaccard; near-miss
# diagnostic (fold counts + mean |coef| + coefficient rank) in BOTH
# directions, so "different heads" can be distinguished from
# "same heads just below the stability threshold".
# ================================================================
_t_cell = time.time()

PUBLISHED_S_HEADS = {
    "mistral": [(8, 16), (9, 23), (12, 9)],
    "gemma2":  [(11, 14), (13, 13), (16, 15)],
    "llama":   [(7, 7), (8, 17)],
    "nemo":    [(8, 9), (10, 24)],
}
S_HEADS_LIST = sorted(PUBLISHED_S_HEADS[ACTIVE_MODEL])
_cfg_heads = sorted((l, h) for l, hs in CFG['heads'].items() for h in hs)
assert _cfg_heads == S_HEADS_LIST, \
    f"Consistency check failed: CFG heads {_cfg_heads} != published table {S_HEADS_LIST}"

K_set, S_set = set(K_HEADS_LIST), set(S_HEADS_LIST)
inter = sorted(K_set & S_set)
union = sorted(K_set | S_set)
jaccard = len(inter) / len(union) if union else float('nan')


def fmt_heads(hs):
    return "{" + ", ".join(f"L{l}H{h}" for l, h in sorted(hs)) + "}"


print("=" * 80)
print(f"OVERLAP ANALYSIS — {CFG['label']}")
print("=" * 80)
print(f"  K-heads (knowledge task, >= 3/5 folds): {fmt_heads(K_set)}")
print(f"  S-heads (published binding heads):      {fmt_heads(S_set)}")
print(f"  intersection: {fmt_heads(inter)}")
print(f"  Jaccard = {jaccard:.3f}   (|intersection| = {len(inter)}, |union| = {len(union)})")


def near_miss_table(cv, heads, cv_label):
    """For each head: how often selected across the 5 folds of this CV,
    its mean |coefficient|, and its rank by mean |coef| among all L*H heads."""
    out = {}
    if cv is None or not heads:
        print(f"\n  (near-miss table for {cv_label} unavailable)")
        return out
    s = cv['bind_avg']['summary']
    if not s:
        print(f"\n  (no CV summary for {cv_label})")
        return out
    counter = {tuple(k): v for k, v in s['head_counter'].items()}
    mc = np.abs(s['mean_coef'])
    order = np.argsort(mc.ravel())[::-1]
    rank_of = {(int(fl // N_HEADS), int(fl % N_HEADS)): r + 1
               for r, fl in enumerate(order)}
    print(f"\n  Near-miss ranks in the {cv_label} CV (bind_avg):")
    print(f"    {'head':>8s}  {'folds/5':>7s}  {'mean|coef|':>10s}  {'rank by |coef|':>14s}"
          f"   (of {N_LAYERS*N_HEADS} heads)")
    for (l, h) in sorted(heads):
        nf = counter.get((l, h), 0)
        coef = float(mc[l, h])
        rk = rank_of[(l, h)]
        out[(l, h)] = {'folds': nf, 'mean_abs_coef': coef, 'coef_rank': rk}
        print(f"    L{l:02d}H{h:02d}  {nf:>7d}  {coef:10.4f}  {rk:>14d}")
    return out


# S-heads under the K discovery (were the binding heads "almost selected" on K?)
near_S_in_K = near_miss_table(cv_K, S_HEADS_LIST, "K-task")
# K-heads under the S discovery (vice versa)
near_K_in_S = near_miss_table(cv_S, K_HEADS_LIST, "S-task")

# ── Layer-distribution comparison ──
k_layers = sorted(l for l, h in K_HEADS_LIST)
s_layers = sorted(l for l, h in S_HEADS_LIST)
print(f"\n  Layer distribution:")
print(f"    S-head layers: {s_layers}  (published S-heads sit in the L7-L16 band across models)")
print(f"    K-head layers: {k_layers if k_layers else '(none)'}")
if K_HEADS_LIST:
    in_band = [(l, h) for (l, h) in K_HEADS_LIST if 7 <= l <= 16]
    print(f"    K-heads inside L7-L16: {len(in_band)}/{len(K_HEADS_LIST)}  "
          f"({fmt_heads(in_band) if in_band else '{}'})")
    print(f"    K-heads outside L7-L16: {fmt_heads([x for x in K_HEADS_LIST if x not in in_band]) if len(in_band) < len(K_HEADS_LIST) else '{}'}")

print(f"\n[cell 7 done in {time.time()-_t_cell:.1f}s]")

## Cell 8 — Causal cross-test: knockout of the NEW K-heads on BOTH tasks (+ GATE: mechanism sanity on 5 prompts)

Completes the 2×2 matrix: the paper has S-heads × {S, K}; this computes K-heads × {S, K}, plus the U→item control on the K task. Same edge-knockout machinery as the reference (R→item = `'B_to_item'`, full identity-span × item-span Cartesian product).

In [ ]:
# ================================================================
# CELL 8 — CAUSAL CROSS-TEST (edge knockout of the K-heads)
# v4: accepts a PRIMARY K-head subset + an optional VARIANT subset.
#     Baselines (S, K) are computed ONCE; only the KO passes repeat
#     per subset. Leak fix unchanged (see v2 note below).
#     Cross-check now asserts on the Δ(K) baseline (the invariant we
#     actually need) rather than per-prompt arrays: bf16 forwards drift
#     per-prompt (~0.1 on Nemo-12B, unbiased), which cancels in the mean
#     but tripped the old 1e-2 per-prompt tolerance. The per-prompt max
#     is still logged as a leak tripwire.
#
# v2 note: edge_knockout leaves its wrapper in
# ALL_ATTENTION_FUNCTIONS._local_mapping on exit and never clears
# _EDGE_REGISTRY; forwards outside a KO context after the first KO are
# phantom-masked. reset_eager_attention() purges both; called after every
# KO block and before every baseline pass.
# ================================================================
_t_cell = time.time()

# ── Subsets to knock out as a group (per model). VARIANT may be None. ──
# K_HEADS_PRIMARY / K_HEADS_VARIANT are defined in the params cell (PER_MODEL,
# consolidated across the four canonical model copies; the llama VARIANT uses
# the final "extra run" value (11, 14) — the first run used (11, 26)).

VARIANTS = [("primary", K_HEADS_PRIMARY.get(ACTIVE_MODEL) or [])]
_var = K_HEADS_VARIANT.get(ACTIVE_MODEL)
if _var:
    VARIANTS.append(("variant", _var))

# Cross-check tolerances. The guard exists to catch the *leak* (deviations of
# order units), not bf16 per-prompt drift (~0.1 on the 12B). So we assert on
# the mean-cancelled Δ(K) baseline, and only log the per-prompt max.
_KO_DELTA_TOL    = 1e-2     # |Δ(K)_KOpath − Δ(K)_cell4| must be tiny
_KO_PERPROMPT_TRIPWIRE = 1.0  # per-prompt max above this = likely a real leak


# reset_eager_attention is defined in the machinery cell above (verbatim khead v4 copy).


# Published S-head knockout rows (instruct; % reduction of |delta| under KO).
PUBLISHED_SHEAD_ROW = {
    "mistral": {"red_B_S": 23.5, "red_B_K": 26.6, "red_A_S": -5.8, "red_A_K": -6.6},
    "gemma2":  {"red_B_S": 13.2, "red_B_K":  4.8, "red_A_S": -5.6, "red_A_K": -2.6},
    "llama":   {"red_B_S": 16.4, "red_B_K": 31.1, "red_A_S": -5.9, "red_A_K": -6.2},
    "nemo":    {"red_B_S": 12.3, "red_B_K": 12.3, "red_A_S": -5.3, "red_A_K": -3.1},
}


# ttest_clustered is imported from common.stats_utils (verbatim copy).


def _to_dict(head_list):
    d = {}
    for l, h in head_list:
        d.setdefault(l, []).append(h)
    return d


print("=" * 80)
print(f"CELL 8: CAUSAL CROSS-TEST — {CFG['label']}")
for tag, hl in VARIANTS:
    print(f"  [{tag}] K-heads under test: {_to_dict(hl) if hl else '(EMPTY)'}")
print("=" * 80)

all_ko_results = {}

if not VARIANTS[0][1]:
    print("\n  !! No primary K-head subset for this model — skipping the causal")
    print("  !! cross-test. The summary (cell 9) will report this explicitly.")
else:
    # ── Positions (built ONCE; shared by every variant) ──
    print("\n  Building positions (binding)...")
    positions = {c: [] for c in conditions}
    for c in conditions:
        for i in range(n_total):
            q_text = data[c][i].split("\n\n")[0]
            oa, ob = extract_options(q_text)
            item = data['items_cult'][i]
            enc = tokenizer(texts_fmt[c][i], return_tensors="pt")
            ids = enc["input_ids"][0].tolist()
            sp = detect_spans(ids, q_text, oa, ob, item, tokenizer, item_required=True)
            if sp is None:
                positions[c].append(None)
            else:
                ap = data['assoc_pos'][i]
                positions[c].append({
                    'item_tokens': sp['item'],
                    'B_tokens': sp['opt_a'] if ap == 'a' else sp['opt_b'],
                    'A_tokens': sp['opt_b'] if ap == 'a' else sp['opt_a']})
            del enc

    print("  Building positions (knowledge)...")
    positions_k = {c: [] for c in k_conditions}
    for c in k_conditions:
        for i in range(n_total):
            q_text = knowledge[c][i].split("\n\n")[0]
            oa, ob = extract_options(q_text)
            item = data['items_cult'][i]
            enc = tokenizer(texts_fmt_k[c][i], return_tensors="pt")
            ids = enc["input_ids"][0].tolist()
            sp = detect_spans(ids, q_text, oa, ob, item, tokenizer, item_required=True)
            if sp is None:
                positions_k[c].append(None)
            else:
                ap = data['assoc_pos'][i]
                positions_k[c].append({
                    'item_tokens': sp['item'],
                    'B_tokens': sp['opt_a'] if ap == 'a' else sp['opt_b'],
                    'A_tokens': sp['opt_b'] if ap == 'a' else sp['opt_a']})
            del enc

    for c in conditions:
        n_none = sum(1 for p in positions[c] if p is None)
        print(f"    binding   {c}: {n_none}/{n_total} pairs without spans (KO -> no-op)")
    for c in k_conditions:
        n_none = sum(1 for p in positions_k[c] if p is None)
        print(f"    knowledge {c}: {n_none}/{n_total} pairs without spans (KO -> no-op)")

    items_arr = np.array(data['items_cult'])

    # ── Baselines (computed ONCE; identical across variants) ──
    reset_eager_attention()
    print(f"\n  [S task] baseline (no KO)...  [{time.time()-_t_cell:.0f}s]")
    s_base = {c: compute_logit_scores_edge(
        model, tokenizer, texts_fmt[c], data[c],
        {}, positions[c], 'B_to_item', option_tokens) for c in conditions}
    delta_S_base = s_base['B_cult'].mean() - s_base['B_unrel'].mean()
    sdiffs_base = s_base['B_cult'] - s_base['B_unrel']
    print(f"    baseline delta(S) = {delta_S_base:.4f}")

    reset_eager_attention()
    print(f"  [K task] baseline (no KO)...  [{time.time()-_t_cell:.0f}s]")
    k_base = {c: compute_logit_scores_edge(
        model, tokenizer, texts_fmt_k[c], knowledge[c],
        {}, positions_k[c], 'B_to_item', option_tokens) for c in k_conditions}
    delta_K_base = k_base['K_cult'].mean() - k_base['K_unrel'].mean()
    kdiffs_base = k_base['K_cult'] - k_base['K_unrel']
    print(f"    baseline delta(K) = {delta_K_base:.4f}")

    # ── Cross-check vs cell-4 K-scores ──
    # Per-prompt deviation is dominated by bf16 drift (unbiased, ~0.1 on the
    # 12B); the invariant we actually rely on is Δ(K), where that drift cancels
    # in the mean. Assert on Δ(K); keep the per-prompt max only as a leak tripwire.
    _dev = max(float(np.abs(k_base['K_cult'] - results_kscore['K_cult']['K']).max()),
               float(np.abs(k_base['K_unrel'] - results_kscore['K_unrel']['K']).max()))
    _delta_K_ref = (results_kscore['K_cult']['K'].mean()
                    - results_kscore['K_unrel']['K'].mean())
    _dev_delta = abs(delta_K_base - _delta_K_ref)
    print(f"    cross-check vs cell-4: per-prompt max|dev| = {_dev:.2e} "
          f"(bf16 drift; tripwire {_KO_PERPROMPT_TRIPWIRE}),  "
          f"|Δ(K) dev| = {_dev_delta:.2e} (tol {_KO_DELTA_TOL})")
    assert _dev < _KO_PERPROMPT_TRIPWIRE, \
        (f"per-prompt deviation {_dev:.3f} exceeds the leak tripwire — this is too "
         f"large for bf16 drift, suspect a residual leak or wrong positions")
    assert _dev_delta < _KO_DELTA_TOL, \
        (f"Δ(K) baseline disagrees with cell 4 by {_dev_delta:.3f} — the KO-path "
         f"baseline is not reproducing cell 4 in the mean; investigate")

    # ── Per-variant KO passes ──
    for tag, k_heads_list in VARIANTS:
        K_DICT = _to_dict(k_heads_list)
        print("\n" + "#" * 80)
        print(f"#  VARIANT [{tag}] — {K_DICT}")
        print("#" * 80)

        KO_CACHE = OUTPUT_DIR / f"{ACTIVE_MODEL}_instruct_khead_knockout_{tag}.pkl"
        ko_results = None
        if KO_CACHE.exists():
            with open(KO_CACHE, "rb") as f:
                _cand = pickle.load(f)
            if _cand.get('k_heads') == k_heads_list and _cand.get('leakfix') is True:
                ko_results = _cand
                print(f"  [cache hit] {KO_CACHE.name} — skipping recomputation")
            else:
                print(f"  [cache stale] heads changed or pre-leakfix — recomputing")

        if ko_results is None:
            # GATE 8.1: KO must change scores for THIS head set (5 K prompts)
            reset_eager_attention()
            idx5 = [i for i in range(n_total) if positions_k['K_cult'][i] is not None][:5]
            texts5 = [texts_fmt_k['K_cult'][i] for i in idx5]
            raw5 = [knowledge['K_cult'][i] for i in idx5]
            pos5 = [positions_k['K_cult'][i] for i in idx5]
            b5 = compute_logit_scores_edge(model, tokenizer, texts5, raw5,
                                           {}, pos5, 'B_to_item', option_tokens)
            k5 = compute_logit_scores_edge(model, tokenizer, texts5, raw5,
                                           K_DICT, pos5, 'B_to_item', option_tokens)
            reset_eager_attention()
            _max_change = float(np.max(np.abs(b5 - k5)))
            assert _max_change > 1e-6, \
                f"GATE 8.1 FAILED [{tag}]: edge knockout changed NO score"
            print(f"  [GATE 8.1 OK] max |change| = {_max_change:.4f} > 0")

            # S task: K-head R->item KO
            reset_eager_attention()
            print(f"  [S task] KO...  [{time.time()-_t_cell:.0f}s]")
            s_ko = {c: compute_logit_scores_edge(
                model, tokenizer, texts_fmt[c], data[c],
                K_DICT, positions[c], 'B_to_item', option_tokens) for c in conditions}
            delta_S_ko = s_ko['B_cult'].mean() - s_ko['B_unrel'].mean()
            sdiffs_ko = s_ko['B_cult'] - s_ko['B_unrel']
            t_S, p_S = ttest_clustered(sdiffs_base, sdiffs_ko, items_arr)
            red_S = (1 - delta_S_ko / delta_S_base) * 100 if abs(delta_S_base) > 1e-10 else 0.0
            print(f"    KO delta(S) = {delta_S_ko:.4f}  reduction = {red_S:+.1f}%  "
                  f"t = {t_S:.3f}, p = {p_S:.6f}")

            # K task: K-head R->item KO
            reset_eager_attention()
            print(f"  [K task] KO...  [{time.time()-_t_cell:.0f}s]")
            k_ko = {c: compute_logit_scores_edge(
                model, tokenizer, texts_fmt_k[c], knowledge[c],
                K_DICT, positions_k[c], 'B_to_item', option_tokens) for c in k_conditions}
            delta_K_ko = k_ko['K_cult'].mean() - k_ko['K_unrel'].mean()
            kdiffs_ko = k_ko['K_cult'] - k_ko['K_unrel']
            t_K, p_K = ttest_clustered(kdiffs_base, kdiffs_ko, items_arr)
            red_K = (1 - delta_K_ko / delta_K_base) * 100 if abs(delta_K_base) > 1e-10 else 0.0
            print(f"    KO delta(K) = {delta_K_ko:.4f}  reduction = {red_K:+.1f}%  "
                  f"t = {t_K:.3f}, p = {p_K:.6f}")

            # K task: U->item control
            print(f"  [K task] U->item KO (control)...  [{time.time()-_t_cell:.0f}s]")
            k_ctrl = {c: compute_logit_scores_edge(
                model, tokenizer, texts_fmt_k[c], knowledge[c],
                K_DICT, positions_k[c], 'A_to_item', option_tokens) for c in k_conditions}
            delta_K_ctrl = k_ctrl['K_cult'].mean() - k_ctrl['K_unrel'].mean()
            kdiffs_ctrl = k_ctrl['K_cult'] - k_ctrl['K_unrel']
            t_Kc, p_Kc = ttest_clustered(kdiffs_base, kdiffs_ctrl, items_arr)
            red_K_ctrl = (1 - delta_K_ctrl / delta_K_base) * 100 if abs(delta_K_base) > 1e-10 else 0.0
            print(f"    U-ctrl delta(K) = {delta_K_ctrl:.4f}  reduction = {red_K_ctrl:+.1f}%  "
                  f"t = {t_Kc:.3f}, p = {p_Kc:.6f}  (should be small)")
            reset_eager_attention()

            ko_results = {
                'k_heads': k_heads_list, 'tag': tag, 'leakfix': True,
                'binding': {
                    'delta_baseline': float(delta_S_base), 'delta_B_ko': float(delta_S_ko),
                    'reduction_B_pct': float(red_S), 't_B': float(t_S), 'p_B': float(p_S),
                    'diffs_base': sdiffs_base.tolist(), 'diffs_B_ko': sdiffs_ko.tolist()},
                'knowledge': {
                    'delta_baseline': float(delta_K_base), 'delta_B_ko': float(delta_K_ko),
                    'delta_A_ko': float(delta_K_ctrl), 'reduction_B_pct': float(red_K),
                    'reduction_A_pct': float(red_K_ctrl), 't_B': float(t_K), 'p_B': float(p_K),
                    't_A': float(t_Kc), 'p_A': float(p_Kc), 'diffs_base': kdiffs_base.tolist(),
                    'diffs_B_ko': kdiffs_ko.tolist(), 'diffs_A_ko': kdiffs_ctrl.tolist()},
                'baseline_crosscheck_perprompt_max': float(_dev),
                'baseline_crosscheck_delta_dev': float(_dev_delta),
            }
            with open(KO_CACHE, "wb") as f:
                pickle.dump(ko_results, f)
            print(f"  Saved {KO_CACHE.name}")

        all_ko_results[tag] = ko_results

# ── 2x2 matrix printout (one block per variant) ──
for tag, ko_results in all_ko_results.items():
    b, k = ko_results['binding'], ko_results['knowledge']
    pub = PUBLISHED_SHEAD_ROW[ACTIVE_MODEL]
    print("\n  " + "=" * 72)
    print(f"  2x2 KNOCKOUT MATRIX [{tag}] — {CFG['label']}  "
          f"({_to_dict(ko_results['k_heads'])})")
    print("  " + "=" * 72)
    print(f"  {'':28s}  {'binding |dS|':>14s}  {'knowledge |dK|':>15s}")
    print(f"  {'K-heads (this subset)':28s}  {b['reduction_B_pct']:+13.1f}%  "
          f"{k['reduction_B_pct']:+14.1f}%")
    print(f"  {'  (p, clustered)':28s}  {b['p_B']:>14.4f}  {k['p_B']:>15.4f}")
    print(f"  {'S-heads (published)':28s}  {pub['red_B_S']:+13.1f}%  {pub['red_B_K']:+14.1f}%")
    print("  " + "-" * 72)
    print(f"  {'K-heads U->item ctrl (K)':28s}  {'':>14s}  "
          f"{k['reduction_A_pct']:+14.1f}%  (p={k['p_A']:.4f})")
    print(f"  {'S-heads ctrl (published)':28s}  {pub['red_A_S']:+13.1f}%  {pub['red_A_K']:+14.1f}%")
    print("  " + "=" * 72)

print(f"\n[cell 8 done in {time.time()-_t_cell:.1f}s]")

## Cell 8b — Per-head knockout (necessity) + optional leave-one-out

Tests each discovered K-head **one by one** (single-head R→item KO, measured on BOTH tasks) — the per-head analogue of the reference Stage-3 `run_loo`, extended to the K task. Optional LOO (drop one head, KO the rest) probes redundancy within the set. Results are cached **after every head**, so the cell is resumable if the runtime dies.

Cost control: `PERHEAD_SET = 'strict'` tests only the ≥4/5-fold heads; `RUN_LOO = True` doubles the cost. Each head ≈ 4 sweeps × 847 prompts.

In [ ]:
# ================================================================
# CELL 8b — PER-HEAD KO (necessity) + optional LEAVE-ONE-OUT
# Same edge-knockout machinery and baselines as cell 8 (group KO).
# ================================================================
_t_cell = time.time()

PERHEAD_SET = "stable"   # "stable" (>= 3/5 folds) or "strict" (>= 4/5, cheaper)
RUN_LOO = False          # leave-one-out (drop one, KO the rest) — doubles the cost

perhead_results = None
if ko_results is None:
    print("  Cell 8 produced no ko_results (empty K-head set) — skipping per-head tests.")
else:
    heads_to_test = list(K_HEADS_LIST if PERHEAD_SET == "stable" else K_HEADS_STRICT)
    n_sweeps = len(heads_to_test) * 4 * (2 if RUN_LOO else 1)
    print(f"  Testing {len(heads_to_test)} heads one by one ({PERHEAD_SET} set, "
          f"LOO={'on' if RUN_LOO else 'off'})")
    print(f"  Cost: {n_sweeps} sweeps x 847 prompts = {n_sweeps*847} forward passes")

    items_arr = np.array(data['items_cult'])
    # Baselines come from the cell-8 group run (same scoring path)
    sdiffs_base = np.array(ko_results['binding']['diffs_base'])
    kdiffs_base = np.array(ko_results['knowledge']['diffs_base'])
    delta_S_base = ko_results['binding']['delta_baseline']
    delta_K_base = ko_results['knowledge']['delta_baseline']

    PERHEAD_CACHE = OUTPUT_DIR / f"{ACTIVE_MODEL}_instruct_khead_perhead.pkl"
    perhead_results = {'single': {}, 'loo': {}, 'heads': heads_to_test,
                       'k_heads_group': K_HEADS_LIST}
    if PERHEAD_CACHE.exists():
        with open(PERHEAD_CACHE, "rb") as f:
            _c = pickle.load(f)
        if _c.get('k_heads_group') == K_HEADS_LIST:
            perhead_results['single'].update(_c.get('single', {}))
            perhead_results['loo'].update(_c.get('loo', {}))
            print(f"  [cache] {len(perhead_results['single'])} single-head + "
                  f"{len(perhead_results['loo'])} LOO entries already cached (resuming)")
        else:
            print(f"  [cache stale] K-head set changed — recomputing all per-head entries")

    def _heads_to_dict(head_list):
        d = {}
        for l, h in head_list:
            d.setdefault(l, []).append(h)
        return d

    def _ko_both_tasks(heads_dict):
        """R->item KO with the given heads, scored on both tasks; clustered t-tests."""
        s_ko = {c: compute_logit_scores_edge(
            model, tokenizer, texts_fmt[c], data[c],
            heads_dict, positions[c], 'B_to_item', option_tokens) for c in conditions}
        k_ko = {c: compute_logit_scores_edge(
            model, tokenizer, texts_fmt_k[c], knowledge[c],
            heads_dict, positions_k[c], 'B_to_item', option_tokens) for c in k_conditions}
        d_S = s_ko['B_cult'].mean() - s_ko['B_unrel'].mean()
        d_K = k_ko['K_cult'].mean() - k_ko['K_unrel'].mean()
        sdiffs = s_ko['B_cult'] - s_ko['B_unrel']
        kdiffs = k_ko['K_cult'] - k_ko['K_unrel']
        t_S, p_S = ttest_clustered(sdiffs_base, sdiffs, items_arr)
        t_K, p_K = ttest_clustered(kdiffs_base, kdiffs, items_arr)
        red_S = (1 - d_S / delta_S_base) * 100 if abs(delta_S_base) > 1e-10 else 0.0
        red_K = (1 - d_K / delta_K_base) * 100 if abs(delta_K_base) > 1e-10 else 0.0
        return {'delta_S': float(d_S), 'delta_K': float(d_K),
                'red_S_pct': float(red_S), 'red_K_pct': float(red_K),
                't_S': float(t_S), 'p_S': float(p_S),
                't_K': float(t_K), 'p_K': float(p_K)}

    def _save_perhead():
        with open(PERHEAD_CACHE, "wb") as f:
            pickle.dump(perhead_results, f)

    print(f"\n  SINGLE-HEAD KO (necessity), R->item, both tasks:")
    print(f"    {'head':>8s}  {'red dS%':>8s}  {'p(S)':>8s}  {'red dK%':>8s}  {'p(K)':>8s}")
    for (l, h) in heads_to_test:
        key = f"L{l}H{h}"
        if key not in perhead_results['single']:
            perhead_results['single'][key] = _ko_both_tasks({l: [h]})
            _save_perhead()  # save after each head -> resumable
        r = perhead_results['single'][key]
        print(f"    {key:>8s}  {r['red_S_pct']:+7.1f}%  {r['p_S']:8.4f}  "
              f"{r['red_K_pct']:+7.1f}%  {r['p_K']:8.4f}   [{time.time()-_t_cell:.0f}s]")

    if RUN_LOO:
        print(f"\n  LEAVE-ONE-OUT (drop one, KO the rest), R->item, both tasks:")
        print(f"    {'dropped':>8s}  {'red dS%':>8s}  {'p(S)':>8s}  {'red dK%':>8s}  {'p(K)':>8s}")
        for (l, h) in heads_to_test:
            key = f"L{l}H{h}"
            if key not in perhead_results['loo']:
                remaining = [(ll, hh) for (ll, hh) in heads_to_test if (ll, hh) != (l, h)]
                perhead_results['loo'][key] = _ko_both_tasks(_heads_to_dict(remaining))
                _save_perhead()
            r = perhead_results['loo'][key]
            print(f"    {key:>8s}  {r['red_S_pct']:+7.1f}%  {r['p_S']:8.4f}  "
                  f"{r['red_K_pct']:+7.1f}%  {r['p_K']:8.4f}   [{time.time()-_t_cell:.0f}s]")

    # group row from cell 8 for comparison
    print(f"\n    {'ALL (group, cell 8)':>20s}  "
          f"S {ko_results['binding']['reduction_B_pct']:+6.1f}%   "
          f"K {ko_results['knowledge']['reduction_B_pct']:+6.1f}%")
    print("\n  Reading: a head is K-specific if its single KO reduces |dK| clearly more")
    print("  than |dS| (and vice versa); sub-additivity vs the group row indicates")
    print("  redundancy within the set (as does a LOO drop with little effect).")

print(f"\n[cell 8b done in {time.time()-_t_cell:.1f}s]")

## Cell 8b (standalone) — per-head knockout, one head at a time (necessity)

Self-contained variant of cell 8b (the "8BB BIS" cell of the canonical copies): it does
NOT depend on the cell-8 group KO — it rebuilds its own positions and baselines, then
knocks out each head of `HEADS_TO_TEST` (params cell, per-model) in isolation, scoring
both tasks. Resumable: the cache is saved after every head.

In [ ]:
# ================================================================
# CELL 8b (STANDALONE) — PER-HEAD KO, une tête à la fois (necessity)
# Version autonome : ne dépend PAS de la cell 8 (group KO).
# Elle reconstruit ses propres positions + baselines, puis teste
# chaque tête isolément (KO d'une seule tête à la fois).
# Même machinerie edge-knockout / reset_eager_attention que la cell 8.
# ================================================================
_t_cell = time.time()

# ── Les têtes à tester, une par une ──
# (per-model list consolidated verbatim in the params cell)
HEADS_TO_TEST = PER_MODEL[MODEL_KEY]["HEADS_TO_TEST"]


# reset_eager_attention is defined in the machinery cell above (verbatim khead v4 copy).


# ttest_clustered is imported from common.stats_utils (verbatim copy).


print("=" * 80)
print(f"CELL 8b STANDALONE: PER-HEAD KO — {CFG['label']}")
print(f"  {len(HEADS_TO_TEST)} têtes testées une par une")
print("=" * 80)

# ── Positions pour l'edge knockout (binding) ──
print("\n  Building positions (binding)...")
positions = {c: [] for c in conditions}
for c in conditions:
    for i in range(n_total):
        q_text = data[c][i].split("\n\n")[0]
        oa, ob = extract_options(q_text)
        item = data['items_cult'][i]
        enc = tokenizer(texts_fmt[c][i], return_tensors="pt")
        ids = enc["input_ids"][0].tolist()
        sp = detect_spans(ids, q_text, oa, ob, item, tokenizer, item_required=True)
        if sp is None:
            positions[c].append(None)
        else:
            assoc_pos = data['assoc_pos'][i]
            B_tokens = sp['opt_a'] if assoc_pos == 'a' else sp['opt_b']
            A_tokens = sp['opt_b'] if assoc_pos == 'a' else sp['opt_a']
            positions[c].append({'item_tokens': sp['item'], 'B_tokens': B_tokens,
                                  'A_tokens': A_tokens})
        del enc

# ── Positions (knowledge) ──
print("  Building positions (knowledge)...")
positions_k = {c: [] for c in k_conditions}
for c in k_conditions:
    for i in range(n_total):
        q_text = knowledge[c][i].split("\n\n")[0]
        oa, ob = extract_options(q_text)
        item = data['items_cult'][i]
        enc = tokenizer(texts_fmt_k[c][i], return_tensors="pt")
        ids = enc["input_ids"][0].tolist()
        sp = detect_spans(ids, q_text, oa, ob, item, tokenizer, item_required=True)
        if sp is None:
            positions_k[c].append(None)
        else:
            assoc_pos = data['assoc_pos'][i]
            B_tokens = sp['opt_a'] if assoc_pos == 'a' else sp['opt_b']
            A_tokens = sp['opt_b'] if assoc_pos == 'a' else sp['opt_a']
            positions_k[c].append({'item_tokens': sp['item'], 'B_tokens': B_tokens,
                                    'A_tokens': A_tokens})
        del enc

for c in conditions:
    n_none = sum(1 for p in positions[c] if p is None)
    print(f"    binding   {c}: {n_none}/{n_total} pairs without spans (KO falls back to no-op)")
for c in k_conditions:
    n_none = sum(1 for p in positions_k[c] if p is None)
    print(f"    knowledge {c}: {n_none}/{n_total} pairs without spans (KO falls back to no-op)")

items_arr = np.array(data['items_cult'])

# ── Baselines (sans KO) ──
reset_eager_attention()
print(f"\n  [S task] baseline (no KO)...  [{time.time()-_t_cell:.0f}s]")
s_base = {c: compute_logit_scores_edge(
    model, tokenizer, texts_fmt[c], data[c],
    {}, positions[c], 'B_to_item', option_tokens) for c in conditions}
delta_S_base = s_base['B_cult'].mean() - s_base['B_unrel'].mean()
sdiffs_base = s_base['B_cult'] - s_base['B_unrel']
print(f"    baseline delta(S) = {delta_S_base:.4f}")

reset_eager_attention()
print(f"  [K task] baseline (no KO)...  [{time.time()-_t_cell:.0f}s]")
k_base = {c: compute_logit_scores_edge(
    model, tokenizer, texts_fmt_k[c], knowledge[c],
    {}, positions_k[c], 'B_to_item', option_tokens) for c in k_conditions}
delta_K_base = k_base['K_cult'].mean() - k_base['K_unrel'].mean()
kdiffs_base = k_base['K_cult'] - k_base['K_unrel']
print(f"    baseline delta(K) = {delta_K_base:.4f}")
reset_eager_attention()


def _ko_both_tasks(heads_dict):
    """KO R->item avec heads_dict, scoré sur les deux tâches; t-tests clusterisés."""
    reset_eager_attention()
    s_ko = {c: compute_logit_scores_edge(
        model, tokenizer, texts_fmt[c], data[c],
        heads_dict, positions[c], 'B_to_item', option_tokens) for c in conditions}
    reset_eager_attention()
    k_ko = {c: compute_logit_scores_edge(
        model, tokenizer, texts_fmt_k[c], knowledge[c],
        heads_dict, positions_k[c], 'B_to_item', option_tokens) for c in k_conditions}
    reset_eager_attention()
    d_S = s_ko['B_cult'].mean() - s_ko['B_unrel'].mean()
    d_K = k_ko['K_cult'].mean() - k_ko['K_unrel'].mean()
    sdiffs = s_ko['B_cult'] - s_ko['B_unrel']
    kdiffs = k_ko['K_cult'] - k_ko['K_unrel']
    t_S, p_S = ttest_clustered(sdiffs_base, sdiffs, items_arr)
    t_K, p_K = ttest_clustered(kdiffs_base, kdiffs, items_arr)
    red_S = (1 - d_S / delta_S_base) * 100 if abs(delta_S_base) > 1e-10 else 0.0
    red_K = (1 - d_K / delta_K_base) * 100 if abs(delta_K_base) > 1e-10 else 0.0
    return {'delta_S': float(d_S), 'delta_K': float(d_K),
            'red_S_pct': float(red_S), 'red_K_pct': float(red_K),
            't_S': float(t_S), 'p_S': float(p_S),
            't_K': float(t_K), 'p_K': float(p_K)}


# ── Cache résumable ──
PERHEAD_CACHE = OUTPUT_DIR / f"{ACTIVE_MODEL}_instruct_perhead_standalone.pkl"
perhead_results = {'single': {}, 'heads': HEADS_TO_TEST,
                   'delta_S_base': float(delta_S_base),
                   'delta_K_base': float(delta_K_base)}
if PERHEAD_CACHE.exists():
    with open(PERHEAD_CACHE, "rb") as f:
        _c = pickle.load(f)
    if _c.get('heads') == HEADS_TO_TEST:
        perhead_results['single'].update(_c.get('single', {}))
        print(f"  [cache] {len(perhead_results['single'])} têtes déjà calculées (reprise)")
    else:
        print(f"  [cache stale] liste de têtes changée — recalcul complet")


def _save_perhead():
    with open(PERHEAD_CACHE, "wb") as f:
        pickle.dump(perhead_results, f)


# ── KO d'une seule tête à la fois ──
print(f"\n  SINGLE-HEAD KO (necessity), R->item, les deux tâches:")
print(f"    {'head':>8s}  {'red dS%':>8s}  {'p(S)':>8s}  {'red dK%':>8s}  {'p(K)':>8s}")
for (l, h) in HEADS_TO_TEST:
    key = f"L{l}H{h}"
    if key not in perhead_results['single']:
        perhead_results['single'][key] = _ko_both_tasks({l: [h]})
        _save_perhead()  # save après chaque tête -> résumable
    r = perhead_results['single'][key]
    print(f"    {key:>8s}  {r['red_S_pct']:+7.1f}%  {r['p_S']:8.4f}  "
          f"{r['red_K_pct']:+7.1f}%  {r['p_K']:8.4f}   [{time.time()-_t_cell:.0f}s]")

print(f"\n  Saved {PERHEAD_CACHE}")
print("\n  Lecture : une tête est K-spécifique si son KO isolé réduit |dK| nettement")
print("  plus que |dS| (et inversement). Compare la somme des réductions isolées au")
print("  KO groupé (cell 8) pour juger la redondance / sous-additivité du set.")

print(f"\n[cell 8b standalone done in {time.time()-_t_cell:.1f}s]")

## Cell 8b (standalone) — heads discovered on the BASE variant

Same standalone per-head knockout, run on `HEADS_TO_TEST_BASE` (params cell): the
head(s) discovered by the base-variant K-CV, tested here on the instruct model. The
canonical nemo copy has no such run (empty list for `"nemo"`, so the loop is a no-op).
NOTE (inherited verbatim from the canonical copies): this cell shares its cache file
`<model>_instruct_perhead_standalone.pkl` with the previous cell — running it marks the
previous cell's cache stale and overwrites it.

In [ ]:
# ================================================================
# CELL 8b (STANDALONE) — PER-HEAD KO, une tête à la fois (necessity)
# Version autonome : ne dépend PAS de la cell 8 (group KO).
# Elle reconstruit ses propres positions + baselines, puis teste
# chaque tête isolément (KO d'une seule tête à la fois).
# Même machinerie edge-knockout / reset_eager_attention que la cell 8.
# ================================================================
_t_cell = time.time()

# ── Les têtes à tester, une par une ──
# (heads discovered on the BASE variant; per-model list consolidated verbatim
#  in the params cell; [] for nemo — the canonical nemo copy has no such run)
HEADS_TO_TEST = PER_MODEL[MODEL_KEY]["HEADS_TO_TEST_BASE"]


# reset_eager_attention is defined in the machinery cell above (verbatim khead v4 copy).


# ttest_clustered is imported from common.stats_utils (verbatim copy).


print("=" * 80)
print(f"CELL 8b STANDALONE: PER-HEAD KO — {CFG['label']}")
print(f"  {len(HEADS_TO_TEST)} têtes testées une par une")
print("=" * 80)

# ── Positions pour l'edge knockout (binding) ──
print("\n  Building positions (binding)...")
positions = {c: [] for c in conditions}
for c in conditions:
    for i in range(n_total):
        q_text = data[c][i].split("\n\n")[0]
        oa, ob = extract_options(q_text)
        item = data['items_cult'][i]
        enc = tokenizer(texts_fmt[c][i], return_tensors="pt")
        ids = enc["input_ids"][0].tolist()
        sp = detect_spans(ids, q_text, oa, ob, item, tokenizer, item_required=True)
        if sp is None:
            positions[c].append(None)
        else:
            assoc_pos = data['assoc_pos'][i]
            B_tokens = sp['opt_a'] if assoc_pos == 'a' else sp['opt_b']
            A_tokens = sp['opt_b'] if assoc_pos == 'a' else sp['opt_a']
            positions[c].append({'item_tokens': sp['item'], 'B_tokens': B_tokens,
                                  'A_tokens': A_tokens})
        del enc

# ── Positions (knowledge) ──
print("  Building positions (knowledge)...")
positions_k = {c: [] for c in k_conditions}
for c in k_conditions:
    for i in range(n_total):
        q_text = knowledge[c][i].split("\n\n")[0]
        oa, ob = extract_options(q_text)
        item = data['items_cult'][i]
        enc = tokenizer(texts_fmt_k[c][i], return_tensors="pt")
        ids = enc["input_ids"][0].tolist()
        sp = detect_spans(ids, q_text, oa, ob, item, tokenizer, item_required=True)
        if sp is None:
            positions_k[c].append(None)
        else:
            assoc_pos = data['assoc_pos'][i]
            B_tokens = sp['opt_a'] if assoc_pos == 'a' else sp['opt_b']
            A_tokens = sp['opt_b'] if assoc_pos == 'a' else sp['opt_a']
            positions_k[c].append({'item_tokens': sp['item'], 'B_tokens': B_tokens,
                                    'A_tokens': A_tokens})
        del enc

for c in conditions:
    n_none = sum(1 for p in positions[c] if p is None)
    print(f"    binding   {c}: {n_none}/{n_total} pairs without spans (KO falls back to no-op)")
for c in k_conditions:
    n_none = sum(1 for p in positions_k[c] if p is None)
    print(f"    knowledge {c}: {n_none}/{n_total} pairs without spans (KO falls back to no-op)")

items_arr = np.array(data['items_cult'])

# ── Baselines (sans KO) ──
reset_eager_attention()
print(f"\n  [S task] baseline (no KO)...  [{time.time()-_t_cell:.0f}s]")
s_base = {c: compute_logit_scores_edge(
    model, tokenizer, texts_fmt[c], data[c],
    {}, positions[c], 'B_to_item', option_tokens) for c in conditions}
delta_S_base = s_base['B_cult'].mean() - s_base['B_unrel'].mean()
sdiffs_base = s_base['B_cult'] - s_base['B_unrel']
print(f"    baseline delta(S) = {delta_S_base:.4f}")

reset_eager_attention()
print(f"  [K task] baseline (no KO)...  [{time.time()-_t_cell:.0f}s]")
k_base = {c: compute_logit_scores_edge(
    model, tokenizer, texts_fmt_k[c], knowledge[c],
    {}, positions_k[c], 'B_to_item', option_tokens) for c in k_conditions}
delta_K_base = k_base['K_cult'].mean() - k_base['K_unrel'].mean()
kdiffs_base = k_base['K_cult'] - k_base['K_unrel']
print(f"    baseline delta(K) = {delta_K_base:.4f}")
reset_eager_attention()


def _ko_both_tasks(heads_dict):
    """KO R->item avec heads_dict, scoré sur les deux tâches; t-tests clusterisés."""
    reset_eager_attention()
    s_ko = {c: compute_logit_scores_edge(
        model, tokenizer, texts_fmt[c], data[c],
        heads_dict, positions[c], 'B_to_item', option_tokens) for c in conditions}
    reset_eager_attention()
    k_ko = {c: compute_logit_scores_edge(
        model, tokenizer, texts_fmt_k[c], knowledge[c],
        heads_dict, positions_k[c], 'B_to_item', option_tokens) for c in k_conditions}
    reset_eager_attention()
    d_S = s_ko['B_cult'].mean() - s_ko['B_unrel'].mean()
    d_K = k_ko['K_cult'].mean() - k_ko['K_unrel'].mean()
    sdiffs = s_ko['B_cult'] - s_ko['B_unrel']
    kdiffs = k_ko['K_cult'] - k_ko['K_unrel']
    t_S, p_S = ttest_clustered(sdiffs_base, sdiffs, items_arr)
    t_K, p_K = ttest_clustered(kdiffs_base, kdiffs, items_arr)
    red_S = (1 - d_S / delta_S_base) * 100 if abs(delta_S_base) > 1e-10 else 0.0
    red_K = (1 - d_K / delta_K_base) * 100 if abs(delta_K_base) > 1e-10 else 0.0
    return {'delta_S': float(d_S), 'delta_K': float(d_K),
            'red_S_pct': float(red_S), 'red_K_pct': float(red_K),
            't_S': float(t_S), 'p_S': float(p_S),
            't_K': float(t_K), 'p_K': float(p_K)}


# ── Cache résumable ──
PERHEAD_CACHE = OUTPUT_DIR / f"{ACTIVE_MODEL}_instruct_perhead_standalone.pkl"
perhead_results = {'single': {}, 'heads': HEADS_TO_TEST,
                   'delta_S_base': float(delta_S_base),
                   'delta_K_base': float(delta_K_base)}
if PERHEAD_CACHE.exists():
    with open(PERHEAD_CACHE, "rb") as f:
        _c = pickle.load(f)
    if _c.get('heads') == HEADS_TO_TEST:
        perhead_results['single'].update(_c.get('single', {}))
        print(f"  [cache] {len(perhead_results['single'])} têtes déjà calculées (reprise)")
    else:
        print(f"  [cache stale] liste de têtes changée — recalcul complet")


def _save_perhead():
    with open(PERHEAD_CACHE, "wb") as f:
        pickle.dump(perhead_results, f)


# ── KO d'une seule tête à la fois ──
print(f"\n  SINGLE-HEAD KO (necessity), R->item, les deux tâches:")
print(f"    {'head':>8s}  {'red dS%':>8s}  {'p(S)':>8s}  {'red dK%':>8s}  {'p(K)':>8s}")
for (l, h) in HEADS_TO_TEST:
    key = f"L{l}H{h}"
    if key not in perhead_results['single']:
        perhead_results['single'][key] = _ko_both_tasks({l: [h]})
        _save_perhead()  # save après chaque tête -> résumable
    r = perhead_results['single'][key]
    print(f"    {key:>8s}  {r['red_S_pct']:+7.1f}%  {r['p_S']:8.4f}  "
          f"{r['red_K_pct']:+7.1f}%  {r['p_K']:8.4f}   [{time.time()-_t_cell:.0f}s]")

print(f"\n  Saved {PERHEAD_CACHE}")
print("\n  Lecture : une tête est K-spécifique si son KO isolé réduit |dK| nettement")
print("  plus que |dS| (et inversement). Compare la somme des réductions isolées au")
print("  KO groupé (cell 8) pour juger la redondance / sous-additivité du set.")

print(f"\n[cell 8b standalone done in {time.time()-_t_cell:.1f}s]")

## Cell 9 — Summary, decision rule, interpretation, save

In [ ]:
# ================================================================
# CELL 9 — SUMMARY + INTERPRETATION + SAVE
# ================================================================
_t_cell = time.time()
from datetime import datetime

DECISION_RULE = (
    "Dissociation is supported only if (i) K-discovery AUC is adequate, (ii) the "
    "K-head and S-head sets differ substantially (low Jaccard, not explained by "
    "near-miss ranks), AND (iii) the 2x2 knockout matrix shows a double dissociation "
    "(K-heads hit K more than S; S-heads hit S more than K in instruct). Otherwise "
    "the data support shared infrastructure.")

pub = PUBLISHED_SHEAD_ROW[ACTIVE_MODEL]

# ── Evaluate the three criteria ──
crit_i = not K_DISCOVERY_WEAK
near_miss_max = max((v['folds'] for v in near_S_in_K.values()), default=0)
crit_ii = bool(K_HEADS_LIST) and (jaccard <= 0.2) and (near_miss_max < 2)
if ko_results is not None:
    red_KH_S = ko_results['binding']['reduction_B_pct']
    red_KH_K = ko_results['knowledge']['reduction_B_pct']
    dd_new = red_KH_K > red_KH_S            # K-heads hit K more than S
    dd_pub = pub['red_B_S'] > pub['red_B_K']  # S-heads hit S more than K (instruct)
    crit_iii = bool(dd_new and dd_pub)
else:
    red_KH_S = red_KH_K = None
    dd_new = dd_pub = None
    crit_iii = False

# ── Interpretation (cautious: weak AUC -> inconclusive, no overclaiming) ──
if not crit_i:
    outcome = "INCONCLUSIVE"
    interp = (
        f"K-head discovery reached mean AUC = {auc_K:.3f} (<= 0.65), so the K-head set "
        f"is not reliable on {CFG['label']} and no overlap conclusion should be drawn "
        f"from this model. The 2x2 matrix and Jaccard are reported for completeness "
        f"only; rerun or rely on the other models before responding to the reviewer.")
elif not K_HEADS_LIST:
    outcome = "INCONCLUSIVE (no stable K-heads)"
    interp = (
        f"Discovery AUC was adequate ({auc_K:.3f}) but no head was selected in >= 3/5 "
        f"folds on the knowledge task, i.e. the K signal is distributed rather than "
        f"concentrated in a small head set. This is weak evidence AGAINST the claim "
        f"that the S-heads are knowledge heads (they were not selected either — see "
        f"the near-miss table, max fold count of an S-head in the K-CV = "
        f"{near_miss_max}/5), but the absence of a comparison set means the formal "
        f"decision rule cannot be applied. Report as inconclusive-leaning-dissociation.")
elif crit_ii and crit_iii:
    outcome = "POSITIVE DISSOCIATION"
    interp = (
        f"All three criteria hold on {CFG['label']}: (i) K-discovery is adequate "
        f"(AUC = {auc_K:.3f}); (ii) the sets differ substantially (Jaccard = "
        f"{jaccard:.3f}; no S-head was selected in more than {near_miss_max}/5 folds "
        f"of the K-CV, so the difference is not a thresholding artifact); (iii) the "
        f"2x2 matrix shows a double dissociation (K-heads: K {red_KH_K:+.1f}% vs S "
        f"{red_KH_S:+.1f}%; published S-heads: S {pub['red_B_S']:+.1f}% vs K "
        f"{pub['red_B_K']:+.1f}%). The data support distinct head populations for "
        f"knowledge retrieval and binding/gating on this model.")
elif jaccard >= 0.5 or near_miss_max >= 3:
    outcome = "SHARED INFRASTRUCTURE"
    interp = (
        f"The independently discovered K-heads substantially coincide with the "
        f"published S-heads on {CFG['label']} (Jaccard = {jaccard:.3f}; max S-head "
        f"fold count in the K-CV = {near_miss_max}/5). Together with the knockout "
        f"matrix (K-heads: S {red_KH_S if red_KH_S is not None else float('nan'):+.1f}% / K "
        f"{red_KH_K if red_KH_K is not None else float('nan'):+.1f}%; published S-heads: S "
        f"{pub['red_B_S']:+.1f}% / K {pub['red_B_K']:+.1f}%), the data support shared "
        f"infrastructure: the same heads carry both the knowledge and the binding "
        f"signal on this model.")
else:
    outcome = "MIXED / PARTIAL"
    interp = (
        f"The criteria are only partially met on {CFG['label']}: Jaccard = "
        f"{jaccard:.3f}, max S-head fold count in the K-CV = {near_miss_max}/5, "
        f"double dissociation = {crit_iii} (K-heads: K "
        f"{red_KH_K if red_KH_K is not None else float('nan'):+.1f}% vs S "
        f"{red_KH_S if red_KH_S is not None else float('nan'):+.1f}%; published S-heads: "
        f"S {pub['red_B_S']:+.1f}% vs K {pub['red_B_K']:+.1f}%). Per the decision rule "
        f"this does NOT establish dissociation; report the numbers as partially "
        f"overlapping infrastructure and compare across models (incl. base variants) "
        f"before drawing a conclusion.")

# ── Print the summary block ──
print("=" * 80)
print(f"K-HEAD DISCOVERY SUMMARY — {CFG['label']} (instruct)")
print("=" * 80)
print(f"\n  K-heads (>= 3/5 folds): {[f'L{l}H{h}' for l, h in K_HEADS_LIST]}")
print(f"  S-heads (published):    {[f'L{l}H{h}' for l, h in S_HEADS_LIST]}")
print(f"  intersection:           {[f'L{l}H{h}' for l, h in inter]}")
print(f"  Jaccard = {jaccard:.3f}")
print(f"  K-discovery mean AUC = {auc_K:.3f}  (weak = {K_DISCOVERY_WEAK})")
print(f"  baselines: |dS| (cell 8) = "
      f"{abs(ko_results['binding']['delta_baseline']) if ko_results else float('nan'):.4f}, "
      f"|dK| (cell 4) = {abs(delta_K):.4f}")
if ko_results is not None:
    print(f"\n  2x2 matrix (% reduction, R->item KO):")
    print(f"    K-heads (new):        S {red_KH_S:+.1f}%   K {red_KH_K:+.1f}%   "
          f"(U-ctrl on K: {ko_results['knowledge']['reduction_A_pct']:+.1f}%)")
    print(f"    S-heads (published):  S {pub['red_B_S']:+.1f}%   K {pub['red_B_K']:+.1f}%")
print(f"\n  DECISION RULE: {DECISION_RULE}")
print(f"\n  Criteria: (i) AUC adequate = {crit_i}   "
      f"(ii) sets differ substantially = {crit_ii}   "
      f"(iii) double dissociation = {crit_iii}")
print(f"\n  OUTCOME: {outcome}")
print(f"\n  INTERPRETATION:\n  {interp}")

# ── Save ──
RESULTS = {
    'meta': {
        'model': ACTIVE_MODEL, 'label': CFG['label'],
        'model_path': CFG['model_path'], 'variant': 'instruct',
        'notebook': 'khead_discovery_instruct',
        'timestamp': datetime.now().isoformat(), 'seed': SEED,
    },
    'k_heads': K_HEADS_LIST,
    'k_heads_strict': K_HEADS_STRICT,
    's_heads_published': S_HEADS_LIST,
    'cv': {'mean_auc': auc_K, 'fold_aucs': fold_aucs,
           'k_discovery_weak': K_DISCOVERY_WEAK,
           'head_counter_K': {tuple(kk): vv for kk, vv in
                              cv_K['bind_avg']['summary'].get('head_counter', {}).items()}
           if cv_K['bind_avg']['summary'] else {}},
    'overlap': {'intersection': inter, 'union': union, 'jaccard': jaccard,
                'near_miss_S_in_K': near_S_in_K, 'near_miss_K_in_S': near_K_in_S,
                'near_miss_max_folds': near_miss_max},
    'baselines': {'delta_K_cell4': float(delta_K),
                  'published_abs_dK': PUBLISHED_ABS_DK[ACTIVE_MODEL]},
    'matrix_2x2': {
        'K_heads': None if ko_results is None else {
            'red_S_pct': red_KH_S, 'red_K_pct': red_KH_K,
            'p_S': ko_results['binding']['p_B'], 'p_K': ko_results['knowledge']['p_B'],
            'red_K_Uctrl_pct': ko_results['knowledge']['reduction_A_pct'],
            'p_K_Uctrl': ko_results['knowledge']['p_A'],
        },
        'S_heads_published': pub,
    },
    'knockout_full': ko_results,
    'criteria': {'i_auc_adequate': crit_i, 'ii_sets_differ': crit_ii,
                 'iii_double_dissociation': crit_iii},
    'decision_rule': DECISION_RULE,
    'outcome': outcome,
    'interpretation': interp,
}

_results_path = OUTPUT_DIR / f"{ACTIVE_MODEL}_instruct_khead_results.pkl"
with open(_results_path, "wb") as f:
    pickle.dump(RESULTS, f)
print(f"\n  Saved: {_results_path}")

print(f"\n[cell 9 done in {time.time()-_t_cell:.1f}s]")